# Тест 1

## Задание 1

Из БВ FRED загрузите ряд `MORTGAGE30US` ([ссылка](https://fred.stlouisfed.org/series/MORTGAGE30US)) c **2000-01-01** по **2025-12-31**. Для **первой разности** этого ряда вычислите коэффициенты _r(3)_ и _rpart(3)_. Ответ округлите до 3-х десятичных знаков.

_r(3)_=Ответ {$а} Вопрос 1

_rpart(3)_=Ответ {$а} Вопрос 1

In [9]:
# Основные библиотеки
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import pandas_datareader.data as web

# Для ARIMA через statsmodels
from statsmodels.tsa.api import ARIMA

# Для ADF и KPSS
from statsmodels.tsa.api import adfuller, kpss

# Для ACF и PACF
from statsmodels.tsa.api import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Для тестов остатков
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

# Для автоматического подбора ARIMA
import pmdarima as pm

# Для графиков
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
# Импорт данных
y = web.DataReader(
    name="MORTGAGE30US",
    data_source="fred",
    start="2000-01-01",
    end="2025-12-31"
).dropna()

# Если нужен Series, а не DataFrame
y = y["MORTGAGE30US"]

# Первая разность
dy = y.diff().dropna()

# ACF и PACF до лага 2 включительно
acf_values = acf(dy, nlags=3)
pacf_values = pacf(dy, nlags=3)

# r(2) и rpart(2)
r3 = acf_values[3]
rpart3 = pacf_values[3]

print("r(3) =", round(r3, 3))
print("rpart(3) =", round(rpart3, 3))

r(3) = 0.017
rpart(3) = -0.005


## Задание 2

Из БВ FRED загрузите ряд `WBAA` ([ссылка](https://fred.stlouisfed.org/series/WBAA)) c **2000-01-01** по **2025-12-31**. Для этого ряда подгоните модель ARIMA порядка (2, 0, 1) **без константы/тренда**. Проведите тест на ARCH-эффекты, число лагов возьмите равным 5. В ответе укажите тестовую статистику и сделайте вывод. Ответ округлите до 3-х десятичных знаков. Уровень значимости 10.0%

Тестовая статистика = Ответ {$а} Вопрос 2

Вывод: Ответ {$а} Вопрос 2 есть гетероскедастичностьнет гетероскедастичности

In [12]:
# Импорт данных
y = web.DataReader(
    name="WBAA",
    data_source="fred",
    start="2000-01-01",
    end="2025-12-31"
).dropna()

# Если нужен Series, а не DataFrame
y = y["WBAA"]

mod = ARIMA(
    y,
    order=(2, 0, 1),
    trend="n",
    missing="drop"
)

res = mod.fit()
# print(res.summary())

# ARCH-тест
lm_stat, lm_pvalue, f_stat, f_pvalue = het_arch(
    res.resid,
    nlags=5
)

print(round(lm_stat, 3))
print(round(lm_pvalue, 3))

# Вывод
if lm_pvalue < 0.05:
    print("есть гетероскедастичность")
else:
    print("нет гетероскедастичности")

96.297
0.0
есть гетероскедастичность


## Задание 3

Из БВ FRED загрузите ряд `WBAA` ([ссылка](https://fred.stlouisfed.org/series/WBAA)) c **2000-01-01** по **2025-12-31**. Для этого ряда подгоните модель ARIMA порядка (1, 0, 2) **без константы/тренда** и укажите её коэффициенты. Ответ округлите до 3-х десятичных знаков.

θ1=Ответ {$а} Вопрос 3

θ2=Ответ {$а} Вопрос 3

In [20]:
# Импорт данных
y = web.DataReader(
    name="WAAA",
    data_source="fred",
    start="2000-01-01",
    end="2025-12-31"
).dropna()

# Если нужен Series, а не DataFrame
y = y["WAAA"]

# ARIMA
mod = ARIMA(
    y,
    order=(1, 0, 2),
    trend="n",
    missing="drop"
)

res = mod.fit()

# MA-коэффициенты
theta1 = res.params["ma.L1"]
theta2 = res.params["ma.L2"]

print("θ1 =", round(theta1, 3))
print("θ2 =", round(theta2, 3))


θ1 = 0.207
θ2 = -0.024


## Задание 4

Из БВ FRED загрузите ряд `WBAA` ([ссылка](https://fred.stlouisfed.org/series/WBAA)) c **2010-01-01** по **2025-12-31**. Для этого ряда подгоните модель ARIMA **оптимального** порядка, используя тест единичного корня `kpss` и информационный критерий `aic`. Параметры `seasonal=False`, `max_p=3`, `max_q=3`. В ответе укажите порядок "оптимальной" модели.

p=Ответ {$а} Вопрос 4

d=Ответ {$а} Вопрос 4

q=Ответ {$а} Вопрос 4

In [24]:
# Загрузка ряда WBAA из FRED
y = web.DataReader(
    name="WBAA",
    data_source="fred",
    start="2010-01-01",
    end="2025-12-31"
).dropna()

# Берём сам ряд как Series
y = y["WBAA"]

# Подбор оптимальной ARIMA
arima = pm.auto_arima(
    y,
    test="kpss",
    information_criterion="aic",
    seasonal=False,
    max_p=3,
    max_q=3,
    suppress_warnings=True,
    error_action="ignore",
    trace=True
)

print("Оптимальный порядок:", arima.order)

p, d, q = arima.order

print("p =", p)
print("d =", d)
print("q =", q)

Performing stepwise search to minimize aic
 ARIMA(2,1,2)(0,0,0)[0] intercept   : AIC=-1571.997, Time=0.17 sec
 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=-1560.265, Time=0.02 sec
 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=-1604.465, Time=0.02 sec
 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=-1604.898, Time=0.03 sec
 ARIMA(0,1,0)(0,0,0)[0]             : AIC=-1562.230, Time=0.02 sec
 ARIMA(1,1,1)(0,0,0)[0] intercept   : AIC=-1603.408, Time=0.03 sec
 ARIMA(0,1,2)(0,0,0)[0] intercept   : AIC=-1603.437, Time=0.04 sec
 ARIMA(1,1,2)(0,0,0)[0] intercept   : AIC=-1600.939, Time=0.09 sec
 ARIMA(0,1,1)(0,0,0)[0]             : AIC=-1606.872, Time=0.02 sec
 ARIMA(1,1,1)(0,0,0)[0]             : AIC=-1605.383, Time=0.02 sec
 ARIMA(0,1,2)(0,0,0)[0]             : AIC=-1605.412, Time=0.03 sec
 ARIMA(1,1,0)(0,0,0)[0]             : AIC=-1606.441, Time=0.02 sec
 ARIMA(1,1,2)(0,0,0)[0]             : AIC=-1602.912, Time=0.05 sec

Best model:  ARIMA(0,1,1)(0,0,0)[0]          
Total fit time: 0.579 seconds
Оптимальн

## Задание 5

Из БВ FRED загрузите ряд `WBAA` ([ссылка](https://fred.stlouisfed.org/series/WBAA)) c **2000-01-01** по **2025-12-31**. Для **первой разности** этого ряда проведите KPSS (выбрав подходящий вариант с константой/трендом) и с **выбором числа лагов по умолчанию**. В ответе укажите тестовую статистику и сделайте вывод. Уровень значимости 1%. Ответ округлите до 3-х десятичных знаков.

Тестовая статистика = Ответ {$а} Вопрос 5

Критическое значение = Ответ {$а} Вопрос 5

Вывод: Ответ {$а} Вопрос 5 есть единичный кореньнет единичного корня

In [25]:
# Загрузка ряда WBAA из FRED
y = web.DataReader(
    name="WBAA",
    data_source="fred",
    start="2010-01-01",
    end="2025-12-31"
).dropna()

# Берём сам ряд как Series
y = y["WBAA"]

# Первая разность
dy = y.diff().dropna()

# С константой и трендом
kpss_result = kpss(
    y,
    regression="ct",
    nlags="auto"
)

# Достать результаты
kpss_stat = kpss_result[0]
pvalue = kpss_result[1]
used_lags = kpss_result[2]
crit_values = kpss_result[3]

print(round(kpss_stat, 3))
print(round(crit_values["5%"], 3))

print("Использованное число лагов:", used_lags)
print("Тестовая статистика:", round(kpss_stat, 3))
print("Критическое значение 1%:", round(crit_values["1%"], 3))
print("p-value:", round(pvalue, 3))

if kpss_stat > crit_values["1%"]:
    print("Вывод: есть единичный корень")
else:
    print("Вывод: нет единичного корня")

0.647
0.146
Использованное число лагов: 18
Тестовая статистика: 0.647
Критическое значение 1%: 0.216
p-value: 0.01
Вывод: есть единичный корень


## Задание 6

Из БВ FRED загрузите ряд `MORTGAGE30US` ([ссылка](https://fred.stlouisfed.org/series/MORTGAGE30US)) c **2000-01-01** по **2025-12-31**. Для этого ряда подгоните модель ARIMA порядка (2, 1, 3) **без константы/тренда**. По подогнанной модели постройте прогноз на 2 периода вперёд. Ответ округлите до 3-х десятичных знаков.

Прогноз на 1 период = Ответ {$а} Вопрос 6

Прогноз на 2 период = Ответ {$а} Вопрос 6

In [26]:
# Импорт данных
y = web.DataReader(
    name="MORTGAGE30US",
    data_source="fred",
    start="2000-01-01",
    end="2025-12-31"
).dropna()

# Если нужен Series, а не DataFrame
y = y["MORTGAGE30US"]

# ARIMA
mod = ARIMA(
    y,
    order=(2, 1, 3),
    trend="n",
    missing="drop"
)

res = mod.fit()
print(res.summary())

# Прогноз
forecast = res.forecast(steps=2)
print(forecast.round(3))

                               SARIMAX Results                                
Dep. Variable:           MORTGAGE30US   No. Observations:                 1357
Model:                 ARIMA(2, 1, 3)   Log Likelihood                1270.028
Date:                Wed, 15 Apr 2026   AIC                          -2528.056
Time:                        02:22:50   BIC                          -2496.783
Sample:                             0   HQIC                         -2516.347
                               - 1357                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1         -0.2863      0.209     -1.372      0.170      -0.695       0.123
ar.L2          0.5092      0.164      3.099      0.002       0.187       0.831
ma.L1          0.3875      0.212      1.824      0.0